In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from tqdm.auto import tqdm
import pandas as pd
import torch


In [2]:
val_data = []
with open("dev.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    val_data.append(json.loads(line))


test_data = []
with open("test.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    test_data.append(json.loads(line))

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200")
model = AutoModelForSeq2SeqLM.from_pretrained("allenai/unifiedqa-v2-t5-3b-1363200", device_map="auto",  torch_dtype=torch.float16)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [16]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    # return set(text.split())
    cleaned = []
    for word in text.split():
        if word.endswith("."):
            word= word[:-1]
        cleaned.append(word)
    return set(cleaned)
            
    


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)

    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))


        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option


    if  not best_option:
        
        best_option = None
        max_overlap = 0
        new_candidate = []
        for z in candidate_words :
            new_candidate.extend( list(z))
        for option in options:
            option_words = preprocess(option)

            new_options =[]
            for z in option_words :
                new_options.extend( list(z))

            overlap = len( set(list(new_candidate)).intersection( set(list(new_candidate))) )

            if overlap > max_overlap:
                max_overlap = overlap
                best_option = option

        return best_option, options.index(best_option)+1

            
    return best_option, options.index(best_option)+1

In [17]:
mannaaa = 'abc'


In [18]:
from string import Template
prompt_template= Template('''Youre a question answering expert. Please answer the question using the context and abbreviations provided. Only provided the correct option (A, B, C, D, E, F, G or H):\n
$question
context: $context      
                                                        
$options
''')


# prompt_template2= Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
# $question
# context: $context      
                                                        
# $options
# $question
# ''')


In [19]:

# data[0]
submission = {"answers":[]}
option_header = ["option A ", "option B ", "option C ", "option D ", "option E ","option F ", "option G ", "option H " ]
map_ans = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
pbar = tqdm(range(len(val_data)))
for example in val_data:
    options = []
    opts = []
    context = ''
    for i, sample in enumerate(example['question']['choices']):
        # print(key)
        
    
        options.append(option_header[i]+ sample['text'])
        opts.append((sample['text'], option_header[i].split("option")[1]))
        context += '\n'+ sample['para']

    # combined_fact = example['combinedfact']

    combined_fact = example["fact1"] + '\n' + example["fact2"]
    # combined_fact = example["fact1"] + '\n' + example["fact2"]
# 


    prompt_sample = prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options))


    input  = tokenizer(prompt_sample, return_tensors="pt").to("cuda")
    # print(input)
    out = model.generate(**input,  max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    answer_only = tokenizer.batch_decode(out)[0].split("<pad>")[1][1:-4]
    ans, id = choose_most_likely_option([op[0] for op in opts], answer_only)
    

    

    # answer_only = text

    

    submission["answers"].append(id)
    pd.DataFrame(submission).to_csv("t5k1_qasc_f1f2.csv")
    
    pbar.set_description(f"Pred: {id:.4f}")
    pbar.update(1)
pbar.close()
# print(answer_only)

  0%|          | 0/926 [00:00<?, ?it/s]

In [20]:
answer_only, options
# choose_most_likely_option(options, answer_only)

('dialysis',
 ['option A Laboratory',
  'option B Lymphocytes',
  'option C saves lives',
  'option D dialysis',
  'option E Lymph fluid',
  'option F dandelions',
  'option G ibuprofen',
  'option H Protein'])

In [8]:
# print(prompt_sample +"\nOption ")
test = prompt_sample +"\nCorrect Option "
tokenizer.pad_token_id = tokenizer.eos_token_id
input  = tokenizer(test, return_tensors="pt").to("cuda")
out = model.generate(**input,  max_new_tokens=50)
text = tokenizer.batch_decode(out)[0]
print(answer_only)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Solution 0:

The correct option is D dialysis.

Renal failure means kidney failure. Kidney failure is a condition in which the kidneys are unable to filter waste products and excess fluid from the blood. Dialysis is a


In [9]:
submission["answers"]


[6,
 7,
 4,
 6,
 2,
 4,
 6,
 5,
 2,
 6,
 1,
 7,
 8,
 3,
 7,
 3,
 4,
 2,
 2,
 8,
 7,
 1,
 5,
 3,
 3,
 3,
 5,
 3,
 5,
 5,
 6,
 3,
 5,
 1,
 1,
 8,
 6,
 8,
 5,
 2,
 4,
 2,
 2,
 4,
 6,
 8,
 8,
 6,
 5,
 3,
 7,
 7,
 2,
 5,
 8,
 1,
 4,
 6,
 7,
 3,
 8,
 3,
 2,
 4,
 3,
 1,
 6,
 3,
 7,
 3,
 7,
 3,
 8,
 5,
 3,
 5,
 6,
 3,
 3,
 6,
 1,
 5,
 1,
 6,
 2,
 5,
 7,
 8,
 7,
 2,
 7,
 2,
 4,
 6,
 5,
 4,
 8,
 1,
 4,
 4,
 1,
 8,
 1,
 7,
 4,
 8,
 5,
 3,
 1,
 1,
 5,
 8,
 3,
 2,
 3,
 2,
 6,
 1,
 7,
 4,
 4,
 3,
 1,
 8,
 5,
 3,
 1,
 4,
 7,
 3,
 3,
 4,
 7,
 2,
 2,
 1,
 7,
 4,
 8,
 7,
 5,
 5,
 7,
 2,
 5,
 3,
 1,
 3,
 6,
 8,
 8,
 3,
 4,
 1,
 4,
 3,
 8,
 1,
 7,
 1,
 4,
 5,
 5,
 5,
 3,
 2,
 4,
 1,
 7,
 7,
 1,
 6,
 8,
 7,
 1,
 4,
 6,
 6,
 5,
 3,
 2,
 2,
 6,
 4,
 3,
 4,
 4,
 2,
 5,
 3,
 3,
 2,
 6,
 3,
 6,
 8,
 8,
 6,
 5,
 5,
 1,
 1,
 1,
 4,
 3,
 3,
 7,
 7,
 5,
 3,
 8,
 8,
 2,
 8,
 6,
 2,
 6,
 2,
 3,
 1,
 3,
 7,
 3,
 5,
 5,
 1,
 8,
 5,
 5,
 5,
 3,
 5,
 7,
 8,
 3,
 4,
 2,
 4,
 8,
 3,
 5,
 8,
 7,
 7,
 3,
 5,
 3,
 2,
 3,
 8,


In [10]:
tokenizer.batch_decode(out)[0]

'Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):\n\nWhat may renal failure be treated with?\ncontext: Kidney failure may be treated with dialysis.\nRenal failure means kidney failure.      \n                                                        \noption A Laboratory\noption B Lymphocytes\noption C saves lives\noption D dialysis\noption E Lymph fluid\noption F dandelions\noption G ibuprofen\noption H Protein\nWhat may renal failure be treated with?\n\nCorrect Option \n\nSolution 0:\n\nThe correct option is D dialysis.\n\nRenal failure means kidney failure. Kidney failure is a condition in which the kidneys are unable to filter waste products and excess fluid from the blood. Dialysis is'

In [11]:
context

"\nRoutine renal laboratory data have been compared with histopathological findings. Company sells dialysis products and list laboratory tests and renal services. Laboratory quantities of oxidizers can be treated. Failure to participate in the laboratory part of the course automatically results in course failure. Laboratory studies reveal a normal hemoglobin and hematocrit and renal panel. Failure to complete the laboratory work is grounds for failure in the course. And what a laboratory it is. Cardiac, pulmonary and renal laboratories begin to function. Field failures are compared with failures in the laboratory. Laboratory findings are also variable, reflecting the primary disease, GN or renal failure.\nWhat role, if any, does BLyS have in regulating B-lymphocyte function. Clemson University researchers have discovered a new way to treat chronic lymphocytic leukemia. Poor B lymphocyte function can recover in patients treated for lead poisoning. T lymphocytes mediate leaflet destructi